In [ ]:
# Compares VideoMAE, VideoMamba and UniFormerV2 and selects the alert threshold (AT).

In [ ]:
# ============================================================
# CELLULE 1 — CONFIGURATION DU NOTEBOOK
# Comparaison UniFormerV2 / VideoMamba / VideoMAE
# et détermination du seuil AT
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score,
)

# ------------------------------------------------------------
# Dossier de sortie
# ------------------------------------------------------------

WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "comparison_and_AT_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Classes RHAD
# L'ordre doit être identique dans les trois modèles
# ------------------------------------------------------------

CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

CLASS_TO_INDEX = {
    class_name: class_index
    for class_index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    class_index: class_name
    for class_name, class_index in CLASS_TO_INDEX.items()
}

NUM_CLASSES = len(CLASS_NAMES)

# On récupère automatiquement l'index de normal.
# On n'écrit pas directement 7 dans le reste du notebook.
NORMAL_CLASS_INDEX = CLASS_TO_INDEX["normal"]

# Toutes les autres classes sont considérées comme anormales
ABNORMAL_CLASS_INDICES = [
    class_index
    for class_index in range(NUM_CLASSES)
    if class_index != NORMAL_CLASS_INDEX
]

EXPECTED_VAL_SIZE = 335
EXPECTED_TEST_SIZE = 335

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

assert NUM_CLASSES == 12
assert NORMAL_CLASS_INDEX == 7
assert len(set(CLASS_NAMES)) == NUM_CLASSES
assert len(ABNORMAL_CLASS_INDICES) == 11

print("Dossier de sortie :", OUTPUT_DIR)
print("Nombre de classes :", NUM_CLASSES)
print("Index de normal   :", NORMAL_CLASS_INDEX)
print("Classes anormales :", ABNORMAL_CLASS_INDICES)

print("\nMapping utilisé :")
for class_index, class_name in INDEX_TO_CLASS.items():
    print(f"{class_index:2d} -> {class_name}")

print("\n config ok")

In [ ]:
# ============================================================
# CELLULE 2 — FONCTIONS POUR DÉTERMINER LE SEUIL AT
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def validate_prediction_dataframe(predictions_df):
    """
    Vérifie que le fichier de prédictions contient les informations
    nécessaires pour rechercher AT.
    """

    required_columns = {
        "true_label",
        "predicted_label",
        "confidence",
    }

    missing_columns = required_columns - set(predictions_df.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes manquantes : "
            + ", ".join(sorted(missing_columns))
        )

    if predictions_df.empty:
        raise ValueError(
            "Le tableau de prédictions est vide."
        )

    valid_indices = set(range(NUM_CLASSES))

    true_indices = set(
        predictions_df["true_label"]
        .astype(int)
        .unique()
    )

    predicted_indices = set(
        predictions_df["predicted_label"]
        .astype(int)
        .unique()
    )

    if not true_indices.issubset(valid_indices):
        raise ValueError(
            f"Indices réels invalides : {true_indices - valid_indices}"
        )

    if not predicted_indices.issubset(valid_indices):
        raise ValueError(
            "Indices prédits invalides : "
            f"{predicted_indices - valid_indices}"
        )

    confidence_values = predictions_df["confidence"].astype(float)

    if not confidence_values.between(0, 1).all():
        raise ValueError(
            "Les valeurs de confidence doivent être comprises entre 0 et 1."
        )

    print(
        f"Tableau valide : {len(predictions_df)} prédictions"
    )


def evaluate_AT_thresholds(
    predictions_df,
    thresholds=None,
    minimum_abnormal_recall=0.90,
):
    """
    Teste plusieurs valeurs de AT.

    Une alerte est produite lorsque :
        predicted_label != normal
        ET confidence >= AT
    """

    validate_prediction_dataframe(predictions_df)

    predictions_df = predictions_df.copy()

    predictions_df["true_label"] = (
        predictions_df["true_label"].astype(int)
    )

    predictions_df["predicted_label"] = (
        predictions_df["predicted_label"].astype(int)
    )

    predictions_df["confidence"] = (
        predictions_df["confidence"].astype(float)
    )

    if thresholds is None:
        thresholds = np.round(
            np.arange(0.20, 0.951, 0.01),
            2,
        )

    # Vérité binaire :
    # 1 = action réellement anormale
    # 0 = action réellement normale
    true_abnormal = (
        predictions_df["true_label"].to_numpy()
        != NORMAL_CLASS_INDEX
    )

    predicted_labels = (
        predictions_df["predicted_label"].to_numpy()
    )

    confidences = predictions_df["confidence"].to_numpy()

    threshold_results = []

    for threshold in thresholds:

        # Alerte seulement si :
        # - la classe prédite n'est pas normal
        # - sa confiance dépasse AT
        predicted_alert = (
            (predicted_labels != NORMAL_CLASS_INDEX)
            & (confidences >= threshold)
        )

        precision = precision_score(
            true_abnormal,
            predicted_alert,
            zero_division=0,
        )

        recall = recall_score(
            true_abnormal,
            predicted_alert,
            zero_division=0,
        )

        f1 = f1_score(
            true_abnormal,
            predicted_alert,
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            true_abnormal,
            predicted_alert,
            labels=[False, True],
        ).ravel()

        # Erreur du modèle :
        # action réellement anormale prédite comme normal
        abnormal_predicted_as_normal = int(
            np.sum(
                true_abnormal
                & (predicted_labels == NORMAL_CLASS_INDEX)
            )
        )

        # Le modèle prédit une action anormale,
        # mais la confiance est insuffisante pour déclencher l'alerte
        rejected_low_confidence = int(
            np.sum(
                (predicted_labels != NORMAL_CLASS_INDEX)
                & (confidences < threshold)
            )
        )

        alert_count = int(predicted_alert.sum())

        threshold_results.append(
            {
                "AT": float(threshold),
                "precision_abnormal": float(precision),
                "recall_abnormal": float(recall),
                "f1_abnormal": float(f1),
                "true_positives": int(tp),
                "false_positives": int(fp),
                "false_negatives": int(fn),
                "true_negatives": int(tn),
                "alert_count": alert_count,
                "rejected_low_confidence": rejected_low_confidence,
                "abnormal_predicted_as_normal":
                    abnormal_predicted_as_normal,
            }
        )

    results_df = pd.DataFrame(threshold_results)

    # Priorité :
    # meilleur F1 parmi les seuils ayant au moins 90 % de rappel
    eligible_results = results_df[
        results_df["recall_abnormal"]
        >= minimum_abnormal_recall
    ].copy()

    selection_rule = (
        f"Meilleur F1 avec rappel anormal "
        f">= {minimum_abnormal_recall:.0%}"
    )

    if eligible_results.empty:
        eligible_results = results_df.copy()

        selection_rule = (
            "Aucun seuil ne respecte le rappel minimal ; "
            "sélection du meilleur F1 disponible"
        )

    best_result = (
        eligible_results
        .sort_values(
            by=[
                "f1_abnormal",
                "recall_abnormal",
                "precision_abnormal",
            ],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

    return results_df, best_result, selection_rule


print(" FONCTIONS DE RECHERCHE DE AT PRÊTES")
print(
    "Aucune valeur finale de AT n'est encore calculée : "
    "il manque les prédictions de validation."
)

In [ ]:
# ============================================================
# CELLULE 3 — TEST DE LA RECHERCHE DE AT
# Données fictives uniquement pour vérifier le code
# ============================================================

np.random.seed(42)

N_FAKE_SAMPLES = 335

# Vraies classes fictives
fake_true_labels = np.random.choice(
    range(NUM_CLASSES),
    size=N_FAKE_SAMPLES,
)

# On crée des prédictions fictives :
# environ 85 % correctes, 15 % incorrectes
fake_predicted_labels = fake_true_labels.copy()

wrong_mask = np.random.rand(N_FAKE_SAMPLES) < 0.15

fake_predicted_labels[wrong_mask] = np.random.choice(
    range(NUM_CLASSES),
    size=wrong_mask.sum(),
)

# Confiance généralement plus élevée lorsque la prédiction est correcte
fake_confidences = np.where(
    fake_predicted_labels == fake_true_labels,
    np.random.uniform(0.55, 0.98, N_FAKE_SAMPLES),
    np.random.uniform(0.20, 0.75, N_FAKE_SAMPLES),
)

fake_predictions_df = pd.DataFrame(
    {
        "true_label": fake_true_labels,
        "predicted_label": fake_predicted_labels,
        "confidence": fake_confidences,
    }
)

print("Exemple des fausses prédictions :")
display(fake_predictions_df.head(10))


# ------------------------------------------------------------
# Recherche de AT
# ------------------------------------------------------------

fake_AT_results, fake_best_AT, fake_selection_rule = (
    evaluate_AT_thresholds(
        fake_predictions_df,
        minimum_abnormal_recall=0.90,
    )
)

print("\nRègle de sélection :")
print(fake_selection_rule)

print("\nMeilleur seuil trouvé sur les données fictives :")
display(fake_best_AT.to_frame().T)

print("\nPremières lignes du tableau des seuils :")
display(fake_AT_results.head(10))

print("\n TEST DE AT TERMINÉ")

In [ ]:
# ============================================================
# CELLULE 4 — VISUALISATION DES MÉTRIQUES SELON AT
# Test provisoire avec les prédictions fictives
# ============================================================

def plot_AT_results(
    results_df,
    best_AT,
    output_path=None,
    title="Recherche du seuil AT",
):
    """
    Trace la précision, le rappel et le F1
    en fonction du seuil AT.
    """

    required_columns = {
        "AT",
        "precision_abnormal",
        "recall_abnormal",
        "f1_abnormal",
    }

    missing_columns = required_columns - set(results_df.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes manquantes pour le graphique : "
            + ", ".join(sorted(missing_columns))
        )

    selected_AT = float(best_AT["AT"])

    plt.figure(figsize=(10, 6))

    plt.plot(
        results_df["AT"],
        results_df["precision_abnormal"],
        label="Précision des alertes",
    )

    plt.plot(
        results_df["AT"],
        results_df["recall_abnormal"],
        label="Rappel des actions anormales",
    )

    plt.plot(
        results_df["AT"],
        results_df["f1_abnormal"],
        label="F1 des alertes",
    )

    plt.axvline(
        selected_AT,
        linestyle="--",
        label=f"AT sélectionné = {selected_AT:.2f}",
    )

    plt.xlabel("Seuil AT")
    plt.ylabel("Score")
    plt.title(title)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
        )
        print("Graphique sauvegardé dans :", output_path)

    plt.show()


FAKE_AT_GRAPH_PATH = OUTPUT_DIR / "fake_AT_curves.png"

plot_AT_results(
    results_df=fake_AT_results,
    best_AT=fake_best_AT,
    output_path=FAKE_AT_GRAPH_PATH,
    title="Test de la recherche de AT sur des données fictives",
)

print("GRAPHIQUE DE TEST CRÉÉ")

In [ ]:
# ============================================================
# CELLULE 5 — CHARGER LES SIX VRAIS FICHIERS DE PRÉDICTIONS
# ============================================================

from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")


def find_prediction_csv(filename: str) -> Path:
    """
    Recherche un CSV dans les datasets Kaggle.
    S'il existe plusieurs copies, sélectionne le chemin
    le plus direct.
    """

    candidates = sorted(
        [
            path
            for path in INPUT_ROOT.rglob(filename)
            if path.is_file()
        ],
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    print(f"\n{filename} :")

    for path in candidates:
        print(" -", path)

    assert candidates, (
        f"{filename} est introuvable dans /kaggle/input."
    )

    selected_path = candidates[0]

    print(
        "Sélectionné :",
        selected_path,
    )

    return selected_path


prediction_files = {
    "VideoMAE": {
        "validation": find_prediction_csv(
            "videomae_validation_predictions.csv"
        ),
        "test": find_prediction_csv(
            "videomae_test_predictions.csv"
        ),
    },
    "VideoMamba Middle": {
        "validation": find_prediction_csv(
            "videomamba_validation_predictions.csv"
        ),
        "test": find_prediction_csv(
            "videomamba_test_predictions.csv"
        ),
    },
    "UniFormerV2": {
        "validation": find_prediction_csv(
            "uniformerv2_validation_predictions.csv"
        ),
        "test": find_prediction_csv(
            "uniformerv2_test_predictions.csv"
        ),
    },
}


required_columns = {
    "true_label",
    "predicted_label",
    "confidence",
    "correct_top1",
    "correct_top5",
}

probability_columns = [
    f"prob_{class_index}"
    for class_index in range(12)
]

prediction_dataframes = {}


for model_name, split_files in prediction_files.items():

    prediction_dataframes[model_name] = {}

    for split_name, csv_path in split_files.items():

        dataframe = pd.read_csv(
            csv_path
        )

        missing_columns = (
            required_columns
            - set(dataframe.columns)
        )

        assert not missing_columns, (
            f"Colonnes manquantes dans {csv_path.name} : "
            f"{sorted(missing_columns)}"
        )

        missing_probability_columns = [
            column
            for column in probability_columns
            if column not in dataframe.columns
        ]

        assert not missing_probability_columns, (
            f"Probabilités manquantes dans {csv_path.name} : "
            f"{missing_probability_columns}"
        )

        assert len(dataframe) == 335, (
            f"{csv_path.name} contient "
            f"{len(dataframe)} vidéos au lieu de 335."
        )

        assert dataframe[
            "true_label"
        ].between(0, 11).all()

        assert dataframe[
            "predicted_label"
        ].between(0, 11).all()

        assert dataframe[
            "confidence"
        ].between(0, 1).all()

        probability_sums = dataframe[
            probability_columns
        ].sum(axis=1)

        assert probability_sums.between(
            0.999,
            1.001,
        ).all(), (
            f"Les probabilités de {csv_path.name} "
            "ne totalisent pas 1."
        )

        prediction_dataframes[
            model_name
        ][split_name] = dataframe


print("\n" + "=" * 80)
print("VRAIES PRÉDICTIONS CHARGÉES")
print("=" * 80)

for model_name in prediction_dataframes:

    validation_df = (
        prediction_dataframes[
            model_name
        ]["validation"]
    )

    test_df = (
        prediction_dataframes[
            model_name
        ]["test"]
    )

    print(
        f"{model_name:<20} | "
        f"validation={len(validation_df)} | "
        f"test={len(test_df)}"
    )

print(
    "\nLES SIX CSV RÉELS SONT PRÊTS "
    "POUR LA COMPARAISON"
)

In [ ]:
# ============================================================
# AUDIT FINAL — UNIFORMERV2, COMPARAISON ET SEUIL AT
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

AUDIT_DIR = Path(
    "/kaggle/working/comparison_and_AT_outputs/final_numerical_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

NORMAL_CLASS_INDEX = 7
SELECTED_AT_AUDIT = 0.354459

PROBABILITY_COLUMNS = [
    f"prob_{index}"
    for index in range(12)
]


# ============================================================
# 1. RAPPORT FINAL UNIFORMERV2 SUR LE TEST
# ============================================================

uniformer_test_df = (
    prediction_dataframes["UniFormerV2"]["test"]
    .copy()
    .reset_index(drop=True)
)

assert len(uniformer_test_df) == 335

uniformer_true = (
    uniformer_test_df["true_label"]
    .astype(int)
    .to_numpy()
)

uniformer_predicted = (
    uniformer_test_df["predicted_label"]
    .astype(int)
    .to_numpy()
)

uniformer_probabilities = (
    uniformer_test_df[PROBABILITY_COLUMNS]
    .astype(float)
    .to_numpy()
)

assert uniformer_true.min() >= 0
assert uniformer_true.max() <= 11
assert uniformer_predicted.min() >= 0
assert uniformer_predicted.max() <= 11

assert np.allclose(
    uniformer_probabilities.sum(axis=1),
    1.0,
    atol=1e-3,
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

uniformer_report_dict = classification_report(
    uniformer_true,
    uniformer_predicted,
    labels=list(range(12)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

uniformer_report_text = classification_report(
    uniformer_true,
    uniformer_predicted,
    labels=list(range(12)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
)

uniformer_report_df = (
    pd.DataFrame(uniformer_report_dict)
    .transpose()
)

uniformer_report_df.to_csv(
    AUDIT_DIR
    / "uniformerv2_final_test_classification_report.csv"
)

(
    AUDIT_DIR
    / "uniformerv2_final_test_classification_report.json"
).write_text(
    json.dumps(
        uniformer_report_dict,
        indent=2,
    ),
    encoding="utf-8",
)

(
    AUDIT_DIR
    / "uniformerv2_final_test_classification_report.txt"
).write_text(
    uniformer_report_text,
    encoding="utf-8",
)


# ------------------------------------------------------------
# Top-1 et Top-5
# ------------------------------------------------------------

uniformer_top1 = accuracy_score(
    uniformer_true,
    uniformer_predicted,
)

if "correct_top5" in uniformer_test_df.columns:
    uniformer_top5 = float(
        uniformer_test_df["correct_top5"]
        .astype(float)
        .mean()
    )
else:
    top5_indices = np.argsort(
        uniformer_probabilities,
        axis=1,
    )[:, -5:]

    uniformer_top5 = float(
        np.mean(
            [
                true_label in predicted_indices
                for true_label, predicted_indices
                in zip(
                    uniformer_true,
                    top5_indices,
                )
            ]
        )
    )


# ------------------------------------------------------------
# Moyennes finales
# ------------------------------------------------------------

uniformer_metrics = {
    "model": "UniFormerV2",
    "split": "test",
    "number_of_videos": 335,
    "top1_accuracy": float(uniformer_top1),
    "top5_accuracy": float(uniformer_top5),

    "macro_precision": float(
        precision_score(
            uniformer_true,
            uniformer_predicted,
            average="macro",
            zero_division=0,
        )
    ),

    "macro_recall": float(
        recall_score(
            uniformer_true,
            uniformer_predicted,
            average="macro",
            zero_division=0,
        )
    ),

    "macro_f1": float(
        f1_score(
            uniformer_true,
            uniformer_predicted,
            average="macro",
            zero_division=0,
        )
    ),

    "weighted_precision": float(
        precision_score(
            uniformer_true,
            uniformer_predicted,
            average="weighted",
            zero_division=0,
        )
    ),

    "weighted_recall": float(
        recall_score(
            uniformer_true,
            uniformer_predicted,
            average="weighted",
            zero_division=0,
        )
    ),

    "weighted_f1": float(
        f1_score(
            uniformer_true,
            uniformer_predicted,
            average="weighted",
            zero_division=0,
        )
    ),
}

(
    AUDIT_DIR
    / "uniformerv2_final_test_metrics.json"
).write_text(
    json.dumps(
        uniformer_metrics,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 2. MATRICE DE CONFUSION UNIFORMERV2
# ============================================================

uniformer_matrix = confusion_matrix(
    uniformer_true,
    uniformer_predicted,
    labels=list(range(12)),
)

uniformer_matrix_df = pd.DataFrame(
    uniformer_matrix,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)

uniformer_matrix_df.to_csv(
    AUDIT_DIR
    / "uniformerv2_final_test_confusion_matrix.csv"
)

plt.figure(figsize=(13, 11))

plt.imshow(
    uniformer_matrix,
    interpolation="nearest",
)

plt.title(
    "UniFormerV2 — Final Test Confusion Matrix"
)

plt.xticks(
    range(12),
    CLASS_NAMES,
    rotation=90,
)

plt.yticks(
    range(12),
    CLASS_NAMES,
)

for row_index in range(12):
    for column_index in range(12):
        value = uniformer_matrix[
            row_index,
            column_index
        ]

        plt.text(
            column_index,
            row_index,
            str(value),
            horizontalalignment="center",
            verticalalignment="center",
            fontsize=8,
        )

plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()

uniformer_matrix_png = (
    AUDIT_DIR
    / "uniformerv2_final_test_confusion_matrix.png"
)

plt.savefig(
    uniformer_matrix_png,
    dpi=220,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 3. GÉNÉRER LES LIGNES LATEX UNIFORMERV2
# ============================================================

latex_lines = []

for class_name in CLASS_NAMES:

    class_metrics = uniformer_report_dict[
        class_name
    ]

    escaped_class_name = (
        class_name.replace("_", r"\_")
    )

    latex_line = (
        f"{escaped_class_name:<26} & "
        f"{class_metrics['precision']:.4f} & "
        f"{class_metrics['recall']:.4f} & "
        f"{class_metrics['f1-score']:.4f} & "
        f"{int(class_metrics['support'])} \\\\"
    )

    latex_lines.append(
        latex_line
    )

latex_lines.extend(
    [
        r"\midrule",
        (
            r"\textbf{Accuracy} & "
            + r"\multicolumn{3}{c}{"
            + f"\\textbf{{{uniformer_top1:.4f}}}"
            + r"} & 335 \\"
        ),
        (
            "Macro average"
            f" & {uniformer_metrics['macro_precision']:.4f}"
            f" & {uniformer_metrics['macro_recall']:.4f}"
            f" & {uniformer_metrics['macro_f1']:.4f}"
            r" & 335 \\"
        ),
        (
            "Weighted average"
            f" & {uniformer_metrics['weighted_precision']:.4f}"
            f" & {uniformer_metrics['weighted_recall']:.4f}"
            f" & {uniformer_metrics['weighted_f1']:.4f}"
            r" & 335 \\"
        ),
    ]
)

latex_text = "\n".join(
    latex_lines
)

(
    AUDIT_DIR
    / "uniformerv2_final_test_latex_rows.txt"
).write_text(
    latex_text,
    encoding="utf-8",
)


# ============================================================
# 4. RECALCULER LE TABLEAU COMPARATIF DE VALIDATION
# ============================================================

def calculate_model_audit_metrics(
    dataframe,
):
    true_labels = (
        dataframe["true_label"]
        .astype(int)
        .to_numpy()
    )

    predicted_labels = (
        dataframe["predicted_label"]
        .astype(int)
        .to_numpy()
    )

    true_abnormal = (
        true_labels != NORMAL_CLASS_INDEX
    ).astype(int)

    predicted_abnormal = (
        predicted_labels != NORMAL_CLASS_INDEX
    ).astype(int)

    true_negative = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_abnormal == 0)
        )
    )

    false_positive = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_abnormal == 1)
        )
    )

    false_negative = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_abnormal == 0)
        )
    )

    true_positive = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_abnormal == 1)
        )
    )

    specificity = (
        true_negative
        / (true_negative + false_positive)
        if (
            true_negative + false_positive
        ) > 0
        else 0.0
    )

    return {
        "top1_accuracy": float(
            accuracy_score(
                true_labels,
                predicted_labels,
            )
        ),

        "top5_accuracy": float(
            dataframe["correct_top5"]
            .astype(float)
            .mean()
        ),

        "macro_f1": float(
            f1_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            )
        ),

        "weighted_f1": float(
            f1_score(
                true_labels,
                predicted_labels,
                average="weighted",
                zero_division=0,
            )
        ),

        "abnormal_precision": float(
            precision_score(
                true_abnormal,
                predicted_abnormal,
                zero_division=0,
            )
        ),

        "abnormal_recall": float(
            recall_score(
                true_abnormal,
                predicted_abnormal,
                zero_division=0,
            )
        ),

        "abnormal_f1": float(
            f1_score(
                true_abnormal,
                predicted_abnormal,
                zero_division=0,
            )
        ),

        "normal_specificity": float(
            specificity
        ),

        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
    }


validation_audit_rows = []

for model_name in [
    "VideoMAE",
    "UniFormerV2",
    "VideoMamba Middle",
]:
    model_metrics = calculate_model_audit_metrics(
        prediction_dataframes[
            model_name
        ]["validation"]
    )

    validation_audit_rows.append(
        {
            "model": model_name,
            **model_metrics,
        }
    )

validation_audit_df = pd.DataFrame(
    validation_audit_rows
)

validation_audit_df.to_csv(
    AUDIT_DIR
    / "validation_model_comparison_recomputed.csv",
    index=False,
)


# ============================================================
# 5. REVÉRIFIER AT SUR VALIDATION ET TEST
# ============================================================

def evaluate_fixed_AT(
    dataframe,
    threshold,
):
    true_labels = (
        dataframe["true_label"]
        .astype(int)
        .to_numpy()
    )

    predicted_labels = (
        dataframe["predicted_label"]
        .astype(int)
        .to_numpy()
    )

    confidences = (
        dataframe["confidence"]
        .astype(float)
        .to_numpy()
    )

    true_abnormal = (
        true_labels != NORMAL_CLASS_INDEX
    ).astype(int)

    predicted_alert = (
        (predicted_labels != NORMAL_CLASS_INDEX)
        & (confidences >= threshold)
    ).astype(int)

    true_positive = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_alert == 1)
        )
    )

    false_positive = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_alert == 1)
        )
    )

    false_negative = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_alert == 0)
        )
    )

    true_negative = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_alert == 0)
        )
    )

    specificity = (
        true_negative
        / (true_negative + false_positive)
        if (
            true_negative + false_positive
        ) > 0
        else 0.0
    )

    return {
        "AT": float(threshold),

        "precision_abnormal": float(
            precision_score(
                true_abnormal,
                predicted_alert,
                zero_division=0,
            )
        ),

        "recall_abnormal": float(
            recall_score(
                true_abnormal,
                predicted_alert,
                zero_division=0,
            )
        ),

        "f1_abnormal": float(
            f1_score(
                true_abnormal,
                predicted_alert,
                zero_division=0,
            )
        ),

        "normal_specificity": float(
            specificity
        ),

        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
    }


AT_validation_audit = evaluate_fixed_AT(
    prediction_dataframes[
        "VideoMAE"
    ]["validation"],
    SELECTED_AT_AUDIT,
)

AT_test_audit = evaluate_fixed_AT(
    prediction_dataframes[
        "VideoMAE"
    ]["test"],
    SELECTED_AT_AUDIT,
)

complete_audit = {
    "uniformerv2_test_metrics": (
        uniformer_metrics
    ),

    "validation_model_comparison": (
        validation_audit_rows
    ),

    "selected_AT": (
        SELECTED_AT_AUDIT
    ),

    "AT_validation_recomputed": (
        AT_validation_audit
    ),

    "AT_test_recomputed": (
        AT_test_audit
    ),
}

(
    AUDIT_DIR
    / "complete_numerical_audit.json"
).write_text(
    json.dumps(
        complete_audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 6. AFFICHAGE FINAL
# ============================================================

print(
    "=" * 80
)

print(
    "UNIFORMERV2 — RAPPORT FINAL TEST"
)

print(
    "=" * 80
)

print(
    uniformer_report_text
)

print(
    "\nMétriques globales UniFormerV2 :"
)

for metric_name, metric_value in (
    uniformer_metrics.items()
):
    if isinstance(
        metric_value,
        float,
    ):
        print(
            f"  {metric_name:<22} : "
            f"{metric_value:.6f}"
        )

print(
    "\nLignes LaTeX :\n"
)

print(
    latex_text
)

print(
    "\n"
    + "=" * 80
)

print(
    "COMPARAISON VALIDATION RECALCULÉE"
)

print(
    "=" * 80
)

display(
    validation_audit_df
)

print(
    "\nAT validation recalculé :"
)

print(
    AT_validation_audit
)

print(
    "\nAT test recalculé :"
)

print(
    AT_test_audit
)

print(
    "\nFichiers enregistrés dans :",
    AUDIT_DIR,
)

print(
    "\nAUDIT NUMÉRIQUE COMPLET"
)

In [ ]:
# ============================================================
# CELLULE 6 — COMPARAISON RÉELLE DES MODÈLES SUR LA VALIDATION
# Sélection du modèle uniquement avec le jeu de validation
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

NORMAL_CLASS_INDEX = 7

COMPARISON_OUTPUT_DIR = Path(
    "/kaggle/working/comparison_and_AT_outputs"
)

COMPARISON_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def calculate_validation_metrics(
    model_name,
    dataframe,
):
    """
    Calcule les métriques de classification multiclasse
    ainsi que les métriques normal/anormal.
    """

    true_labels = dataframe[
        "true_label"
    ].astype(int).to_numpy()

    predicted_labels = dataframe[
        "predicted_label"
    ].astype(int).to_numpy()

    # --------------------------------------------------------
    # Métriques multiclasse
    # --------------------------------------------------------

    top1_accuracy = accuracy_score(
        true_labels,
        predicted_labels,
    )

    if "correct_top5" in dataframe.columns:
        top5_accuracy = float(
            dataframe[
                "correct_top5"
            ].astype(float).mean()
        )
    else:
        probability_columns = [
            f"prob_{class_index}"
            for class_index in range(12)
        ]

        probabilities = dataframe[
            probability_columns
        ].to_numpy()

        top5_predictions = np.argsort(
            probabilities,
            axis=1,
        )[:, -5:]

        top5_accuracy = np.mean(
            [
                true_label in row
                for true_label, row in zip(
                    true_labels,
                    top5_predictions,
                )
            ]
        )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0,
    )

    # --------------------------------------------------------
    # Conversion binaire :
    # 0 = normal
    # 1 = anormal
    # --------------------------------------------------------

    true_abnormal = (
        true_labels
        != NORMAL_CLASS_INDEX
    ).astype(int)

    predicted_abnormal = (
        predicted_labels
        != NORMAL_CLASS_INDEX
    ).astype(int)

    abnormal_precision = precision_score(
        true_abnormal,
        predicted_abnormal,
        zero_division=0,
    )

    abnormal_recall = recall_score(
        true_abnormal,
        predicted_abnormal,
        zero_division=0,
    )

    abnormal_f1 = f1_score(
        true_abnormal,
        predicted_abnormal,
        zero_division=0,
    )

    # Nombre d’actions réellement anormales
    # prédites comme normal.
    abnormal_predicted_as_normal = int(
        np.sum(
            (true_labels != NORMAL_CLASS_INDEX)
            &
            (predicted_labels == NORMAL_CLASS_INDEX)
        )
    )

    # Nombre de vidéos normales prédites comme anormales.
    normal_predicted_as_abnormal = int(
        np.sum(
            (true_labels == NORMAL_CLASS_INDEX)
            &
            (predicted_labels != NORMAL_CLASS_INDEX)
        )
    )

    true_normal_count = int(
        np.sum(
            true_labels
            == NORMAL_CLASS_INDEX
        )
    )

    true_abnormal_count = int(
        np.sum(
            true_labels
            != NORMAL_CLASS_INDEX
        )
    )

    normal_specificity = (
        1.0
        - (
            normal_predicted_as_abnormal
            / true_normal_count
        )
        if true_normal_count > 0
        else 0.0
    )

    return {
        "model": model_name,
        "number_of_videos": len(dataframe),
        "top1_accuracy": top1_accuracy,
        "top5_accuracy": top5_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "abnormal_precision": abnormal_precision,
        "abnormal_recall": abnormal_recall,
        "abnormal_f1": abnormal_f1,
        "normal_specificity": normal_specificity,
        "true_abnormal_videos": true_abnormal_count,
        "true_normal_videos": true_normal_count,
        "abnormal_predicted_as_normal": (
            abnormal_predicted_as_normal
        ),
        "normal_predicted_as_abnormal": (
            normal_predicted_as_abnormal
        ),
        "mean_confidence": float(
            dataframe[
                "confidence"
            ].mean()
        ),
    }


# ============================================================
# 1. CALCULER LES MÉTRIQUES DES TROIS MODÈLES
# ============================================================

validation_results = []

for model_name, split_data in (
    prediction_dataframes.items()
):
    validation_dataframe = (
        split_data["validation"]
        .copy()
        .reset_index(drop=True)
    )

    assert len(
        validation_dataframe
    ) == 335

    metrics = calculate_validation_metrics(
        model_name=model_name,
        dataframe=validation_dataframe,
    )

    validation_results.append(
        metrics
    )


validation_comparison_df = pd.DataFrame(
    validation_results
)


# ============================================================
# 2. CLASSER LES MODÈLES
#
# Critère principal : Macro F1
# Puis :
# - rappel des actions anormales ;
# - accuracy Top-1 ;
# - Weighted F1.
# ============================================================

validation_comparison_df = (
    validation_comparison_df
    .sort_values(
        by=[
            "macro_f1",
            "abnormal_recall",
            "top1_accuracy",
            "weighted_f1",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

validation_comparison_df.insert(
    0,
    "rank",
    range(
        1,
        len(validation_comparison_df) + 1,
    ),
)

SELECTED_MODEL_NAME = str(
    validation_comparison_df.loc[
        0,
        "model",
    ]
)

SELECTED_VALIDATION_DF = (
    prediction_dataframes[
        SELECTED_MODEL_NAME
    ]["validation"]
    .copy()
    .reset_index(drop=True)
)

SELECTED_TEST_DF = (
    prediction_dataframes[
        SELECTED_MODEL_NAME
    ]["test"]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. SAUVEGARDER LES RÉSULTATS
# ============================================================

comparison_csv_path = (
    COMPARISON_OUTPUT_DIR
    / "validation_model_comparison.csv"
)

validation_comparison_df.to_csv(
    comparison_csv_path,
    index=False,
)

selection_summary = {
    "selection_split": "validation",
    "primary_selection_metric": "macro_f1",
    "tie_breaking_metrics": [
        "abnormal_recall",
        "top1_accuracy",
        "weighted_f1",
    ],
    "selected_model": SELECTED_MODEL_NAME,
    "normal_class_index": NORMAL_CLASS_INDEX,
    "number_of_validation_videos": 335,
    "number_of_test_videos": 335,
}

selection_json_path = (
    COMPARISON_OUTPUT_DIR
    / "selected_model.json"
)

selection_json_path.write_text(
    json.dumps(
        selection_summary,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 4. AFFICHAGE
# ============================================================

display_columns = [
    "rank",
    "model",
    "top1_accuracy",
    "top5_accuracy",
    "macro_f1",
    "weighted_f1",
    "abnormal_precision",
    "abnormal_recall",
    "abnormal_f1",
    "normal_specificity",
    "abnormal_predicted_as_normal",
    "normal_predicted_as_abnormal",
]

display_df = validation_comparison_df[
    display_columns
].copy()

percentage_columns = [
    "top1_accuracy",
    "top5_accuracy",
    "macro_f1",
    "weighted_f1",
    "abnormal_precision",
    "abnormal_recall",
    "abnormal_f1",
    "normal_specificity",
]

for column in percentage_columns:
    display_df[column] = (
        100
        * display_df[column]
    ).round(3)

print(
    "=" * 95
)

print(
    "COMPARAISON DES MODÈLES — "
    "VALIDATION UNIQUEMENT"
)

print(
    "=" * 95
)

display(
    display_df
)

print(
    "\nModèle sélectionné :",
    SELECTED_MODEL_NAME,
)

print(
    "Critère principal  : Macro F1 validation"
)

print(
    "CSV sauvegardé     :",
    comparison_csv_path,
)

print(
    "Résumé sauvegardé  :",
    selection_json_path,
)

print(
    "\nMODÈLE SÉLECTIONNÉ SANS UTILISER "
    "LES RÉSULTATS DU TEST"
)

In [ ]:
# ============================================================
# CELLULE 7 — RECHERCHE DU SEUIL AT SUR LA VRAIE VALIDATION
# Modèle sélectionné : VideoMAE
# Le test n'est toujours pas utilisé
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

NORMAL_CLASS_INDEX = 7

# On exige de conserver au minimum 95 % des actions anormales.
# Les actions manquées sont plus graves que les fausses alertes.
MIN_ABNORMAL_RECALL = 0.95

AT_OUTPUT_DIR = Path(
    "/kaggle/working/comparison_and_AT_outputs"
)

AT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. PRÉPARER LES DONNÉES DE VALIDATION VIDEOMAE
# ============================================================

validation_df = (
    SELECTED_VALIDATION_DF
    .copy()
    .reset_index(drop=True)
)

assert SELECTED_MODEL_NAME == "VideoMAE"
assert len(validation_df) == 335

true_labels = (
    validation_df["true_label"]
    .astype(int)
    .to_numpy()
)

predicted_labels = (
    validation_df["predicted_label"]
    .astype(int)
    .to_numpy()
)

confidences = (
    validation_df["confidence"]
    .astype(float)
    .to_numpy()
)

true_abnormal = (
    true_labels != NORMAL_CLASS_INDEX
).astype(int)


# ============================================================
# 2. CRÉER LES SEUILS CANDIDATS
# ============================================================

unique_confidences = np.sort(
    np.unique(confidences)
)

midpoints = (
    unique_confidences[:-1]
    + unique_confidences[1:]
) / 2

AT_THRESHOLDS = np.unique(
    np.concatenate(
        [
            np.array([0.0]),
            unique_confidences,
            midpoints,
            np.array([1.0]),
        ]
    )
)

AT_THRESHOLDS = AT_THRESHOLDS[
    (AT_THRESHOLDS >= 0.0)
    & (AT_THRESHOLDS <= 1.0)
]

print(
    "Nombre de seuils évalués :",
    len(AT_THRESHOLDS),
)


# ============================================================
# 3. ÉVALUER CHAQUE SEUIL
#
# Une alerte est déclenchée lorsque :
# predicted_label != normal ET confidence >= AT
# ============================================================

AT_rows = []

for AT in AT_THRESHOLDS:

    predicted_alert = (
        (predicted_labels != NORMAL_CLASS_INDEX)
        & (confidences >= AT)
    ).astype(int)

    true_positive = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_alert == 1)
        )
    )

    false_positive = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_alert == 1)
        )
    )

    false_negative = int(
        np.sum(
            (true_abnormal == 1)
            & (predicted_alert == 0)
        )
    )

    true_negative = int(
        np.sum(
            (true_abnormal == 0)
            & (predicted_alert == 0)
        )
    )

    precision_abnormal = precision_score(
        true_abnormal,
        predicted_alert,
        zero_division=0,
    )

    recall_abnormal = recall_score(
        true_abnormal,
        predicted_alert,
        zero_division=0,
    )

    f1_abnormal = f1_score(
        true_abnormal,
        predicted_alert,
        zero_division=0,
    )

    normal_specificity = (
        true_negative
        / (true_negative + false_positive)
        if (
            true_negative
            + false_positive
        ) > 0
        else 0.0
    )

    alert_rate = float(
        predicted_alert.mean()
    )

    AT_rows.append(
        {
            "AT": float(AT),
            "precision_abnormal": float(
                precision_abnormal
            ),
            "recall_abnormal": float(
                recall_abnormal
            ),
            "f1_abnormal": float(
                f1_abnormal
            ),
            "normal_specificity": float(
                normal_specificity
            ),
            "true_positive": true_positive,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_negative": true_negative,
            "alert_rate": alert_rate,
        }
    )


real_AT_results = pd.DataFrame(
    AT_rows
)


# ============================================================
# 4. SÉLECTIONNER AT
#
# 1. Rappel anormal >= 95 %
# 2. Meilleur F1
# 3. Meilleure spécificité
# 4. Meilleure précision
# 5. Seuil le plus élevé en cas d'égalité
# ============================================================

eligible_AT_results = real_AT_results[
    real_AT_results["recall_abnormal"]
    >= MIN_ABNORMAL_RECALL
].copy()

assert not eligible_AT_results.empty, (
    "Aucun seuil ne respecte le rappel minimal."
)

eligible_AT_results = (
    eligible_AT_results
    .sort_values(
        by=[
            "f1_abnormal",
            "normal_specificity",
            "precision_abnormal",
            "AT",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

REAL_BEST_AT = (
    eligible_AT_results
    .iloc[0]
    .to_dict()
)

SELECTED_AT = float(
    REAL_BEST_AT["AT"]
)


# ============================================================
# 5. SAUVEGARDER LES RÉSULTATS
# ============================================================

AT_RESULTS_PATH = (
    AT_OUTPUT_DIR
    / "videomae_validation_AT_search.csv"
)

real_AT_results.to_csv(
    AT_RESULTS_PATH,
    index=False,
)

AT_SUMMARY = {
    "selected_model": SELECTED_MODEL_NAME,
    "selection_split": "validation",
    "normal_class_index": NORMAL_CLASS_INDEX,
    "minimum_abnormal_recall": (
        MIN_ABNORMAL_RECALL
    ),
    "selected_AT": SELECTED_AT,
    "validation_metrics_at_selected_AT": {
        key: (
            int(value)
            if key in {
                "true_positive",
                "false_positive",
                "false_negative",
                "true_negative",
            }
            else float(value)
        )
        for key, value
        in REAL_BEST_AT.items()
        if key != "AT"
    },
}

AT_SUMMARY_PATH = (
    AT_OUTPUT_DIR
    / "selected_AT.json"
)

AT_SUMMARY_PATH.write_text(
    json.dumps(
        AT_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 6. GRAPHIQUE RÉEL
# ============================================================

REAL_AT_GRAPH_PATH = (
    AT_OUTPUT_DIR
    / "videomae_real_AT_curves.png"
)

plt.figure(figsize=(10, 6))

plt.plot(
    real_AT_results["AT"],
    real_AT_results[
        "precision_abnormal"
    ],
    label="Précision des alertes",
)

plt.plot(
    real_AT_results["AT"],
    real_AT_results[
        "recall_abnormal"
    ],
    label="Rappel des actions anormales",
)

plt.plot(
    real_AT_results["AT"],
    real_AT_results[
        "f1_abnormal"
    ],
    label="F1 des alertes",
)

plt.plot(
    real_AT_results["AT"],
    real_AT_results[
        "normal_specificity"
    ],
    label="Spécificité des vidéos normales",
)

plt.axhline(
    MIN_ABNORMAL_RECALL,
    linestyle=":",
    label=(
        "Rappel minimal = "
        f"{MIN_ABNORMAL_RECALL:.0%}"
    ),
)

plt.axvline(
    SELECTED_AT,
    linestyle="--",
    label=(
        "AT sélectionné = "
        f"{SELECTED_AT:.4f}"
    ),
)

plt.xlabel("Seuil AT")
plt.ylabel("Score")
plt.title(
    "Recherche réelle du seuil AT — "
    "VideoMAE validation"
)

plt.ylim(0, 1.05)
plt.grid(
    True,
    alpha=0.3,
)

plt.legend()
plt.tight_layout()

plt.savefig(
    REAL_AT_GRAPH_PATH,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 7. AFFICHAGE FINAL
# ============================================================

print("\n" + "=" * 80)
print("SEUIL AT SÉLECTIONNÉ — VALIDATION VIDEOMAE")
print("=" * 80)

print(
    f"AT                         : "
    f"{SELECTED_AT:.6f}"
)

print(
    f"Précision des alertes      : "
    f"{REAL_BEST_AT['precision_abnormal']:.3%}"
)

print(
    f"Rappel actions anormales   : "
    f"{REAL_BEST_AT['recall_abnormal']:.3%}"
)

print(
    f"F1 des alertes             : "
    f"{REAL_BEST_AT['f1_abnormal']:.3%}"
)

print(
    f"Spécificité vidéos normales: "
    f"{REAL_BEST_AT['normal_specificity']:.3%}"
)

print(
    "Actions anormales détectées :",
    int(REAL_BEST_AT["true_positive"]),
)

print(
    "Actions anormales manquées  :",
    int(REAL_BEST_AT["false_negative"]),
)

print(
    "Fausses alertes normales     :",
    int(REAL_BEST_AT["false_positive"]),
)

print(
    "Vidéos normales sans alerte  :",
    int(REAL_BEST_AT["true_negative"]),
)

print(
    "\nRecherche complète :",
    AT_RESULTS_PATH,
)

print(
    "AT sauvegardé      :",
    AT_SUMMARY_PATH,
)

print(
    "Graphique réel     :",
    REAL_AT_GRAPH_PATH,
)

print(
    "\nAT SÉLECTIONNÉ UNIQUEMENT "
    "SUR LA VALIDATION"
)

In [ ]:
# ============================================================
# CELLULE 8 — ÉVALUATION FINALE VIDEOMAE + AT SUR LE TEST
# Le modèle et AT sont déjà figés avec la validation
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

NORMAL_CLASS_INDEX = 7

FINAL_OUTPUT_DIR = Path(
    "/kaggle/working/comparison_and_AT_outputs"
)

FINAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. DONNÉES DU TEST DU MODÈLE SÉLECTIONNÉ
# ============================================================

test_df = (
    SELECTED_TEST_DF
    .copy()
    .reset_index(drop=True)
)

assert SELECTED_MODEL_NAME == "VideoMAE"
assert len(test_df) == 335
assert 0.0 <= SELECTED_AT <= 1.0

true_labels = (
    test_df["true_label"]
    .astype(int)
    .to_numpy()
)

predicted_labels = (
    test_df["predicted_label"]
    .astype(int)
    .to_numpy()
)

confidences = (
    test_df["confidence"]
    .astype(float)
    .to_numpy()
)

true_abnormal = (
    true_labels != NORMAL_CLASS_INDEX
).astype(int)


# ============================================================
# 2. DÉCISIONS AVANT ET APRÈS AT
# ============================================================

# Décision brute du modèle
baseline_alert = (
    predicted_labels != NORMAL_CLASS_INDEX
).astype(int)

# Décision finale du système
final_alert = (
    (predicted_labels != NORMAL_CLASS_INDEX)
    &
    (confidences >= SELECTED_AT)
).astype(int)


# ============================================================
# 3. FONCTION DE CALCUL DES MÉTRIQUES D’ALERTE
# ============================================================

def calculate_alert_metrics(
    true_binary,
    predicted_binary,
):
    matrix = confusion_matrix(
        true_binary,
        predicted_binary,
        labels=[0, 1],
    )

    true_negative = int(matrix[0, 0])
    false_positive = int(matrix[0, 1])
    false_negative = int(matrix[1, 0])
    true_positive = int(matrix[1, 1])

    specificity = (
        true_negative
        / (true_negative + false_positive)
        if (
            true_negative
            + false_positive
        ) > 0
        else 0.0
    )

    return {
        "accuracy": float(
            accuracy_score(
                true_binary,
                predicted_binary,
            )
        ),
        "precision_abnormal": float(
            precision_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),
        "recall_abnormal": float(
            recall_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),
        "f1_abnormal": float(
            f1_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),
        "normal_specificity": float(
            specificity
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
    }


baseline_metrics = calculate_alert_metrics(
    true_abnormal,
    baseline_alert,
)

final_AT_metrics = calculate_alert_metrics(
    true_abnormal,
    final_alert,
)


# ============================================================
# 4. MÉTRIQUES MULTICLASSES DU MODÈLE
# ============================================================

multiclass_metrics = {
    "top1_accuracy": float(
        accuracy_score(
            true_labels,
            predicted_labels,
        )
    ),
    "top5_accuracy": float(
        test_df["correct_top5"]
        .astype(float)
        .mean()
    ),
    "macro_f1": float(
        f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        )
    ),
    "weighted_f1": float(
        f1_score(
            true_labels,
            predicted_labels,
            average="weighted",
            zero_division=0,
        )
    ),
}


# ============================================================
# 5. CRÉER LE CSV DES DÉCISIONS FINALES
# ============================================================

test_df["true_is_abnormal"] = (
    true_abnormal
)

test_df["baseline_alert"] = (
    baseline_alert
)

test_df["selected_AT"] = (
    SELECTED_AT
)

test_df["alert_triggered"] = (
    final_alert
)

test_df["alert_correct"] = (
    final_alert
    == true_abnormal
)

test_df["alert_outcome"] = np.select(
    [
        (true_abnormal == 1) & (final_alert == 1),
        (true_abnormal == 0) & (final_alert == 1),
        (true_abnormal == 1) & (final_alert == 0),
        (true_abnormal == 0) & (final_alert == 0),
    ],
    [
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative",
    ],
    default="unknown",
)

FINAL_DECISIONS_PATH = (
    FINAL_OUTPUT_DIR
    / "videomae_test_final_alert_decisions.csv"
)

test_df.to_csv(
    FINAL_DECISIONS_PATH,
    index=False,
)


# ============================================================
# 6. SAUVEGARDER LE RÉSUMÉ FINAL
# ============================================================

final_summary = {
    "selected_model": SELECTED_MODEL_NAME,
    "selected_using": "validation_macro_f1",
    "selected_AT": float(SELECTED_AT),
    "AT_selected_using": "validation_only",
    "normal_class_index": NORMAL_CLASS_INDEX,
    "test_number_of_videos": len(test_df),
    "test_multiclass_metrics": multiclass_metrics,
    "test_alert_metrics_without_AT": baseline_metrics,
    "test_alert_metrics_with_AT": final_AT_metrics,
}

FINAL_SUMMARY_PATH = (
    FINAL_OUTPUT_DIR
    / "final_videomae_AT_test_results.json"
)

FINAL_SUMMARY_PATH.write_text(
    json.dumps(
        final_summary,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 7. TABLEAU AVANT / APRÈS AT
# ============================================================

comparison_rows = []

for system_name, metrics in [
    ("VideoMAE sans AT", baseline_metrics),
    (
        f"VideoMAE avec AT={SELECTED_AT:.6f}",
        final_AT_metrics,
    ),
]:
    comparison_rows.append(
        {
            "system": system_name,
            **metrics,
        }
    )

test_AT_comparison_df = pd.DataFrame(
    comparison_rows
)

TEST_COMPARISON_PATH = (
    FINAL_OUTPUT_DIR
    / "test_alert_metrics_before_after_AT.csv"
)

test_AT_comparison_df.to_csv(
    TEST_COMPARISON_PATH,
    index=False,
)


# ============================================================
# 8. MATRICE DE CONFUSION BINAIRE FINALE
# ============================================================

final_matrix = confusion_matrix(
    true_abnormal,
    final_alert,
    labels=[0, 1],
)

FINAL_MATRIX_CSV_PATH = (
    FINAL_OUTPUT_DIR
    / "final_test_alert_confusion_matrix.csv"
)

pd.DataFrame(
    final_matrix,
    index=[
        "true_normal",
        "true_abnormal",
    ],
    columns=[
        "predicted_no_alert",
        "predicted_alert",
    ],
).to_csv(
    FINAL_MATRIX_CSV_PATH
)

FINAL_MATRIX_PNG_PATH = (
    FINAL_OUTPUT_DIR
    / "final_test_alert_confusion_matrix.png"
)

plt.figure(figsize=(6, 5))

plt.imshow(
    final_matrix,
    interpolation="nearest",
)

plt.title(
    "Matrice de confusion finale\n"
    f"VideoMAE + AT={SELECTED_AT:.4f}"
)

plt.xticks(
    [0, 1],
    [
        "Pas d’alerte",
        "Alerte",
    ],
)

plt.yticks(
    [0, 1],
    [
        "Réel normal",
        "Réel anormal",
    ],
)

for row_index in range(2):
    for column_index in range(2):
        plt.text(
            column_index,
            row_index,
            str(
                final_matrix[
                    row_index,
                    column_index,
                ]
            ),
            horizontalalignment="center",
            verticalalignment="center",
        )

plt.xlabel("Décision du système")
plt.ylabel("Vérité terrain")
plt.tight_layout()

plt.savefig(
    FINAL_MATRIX_PNG_PATH,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 9. AFFICHAGE FINAL
# ============================================================

display_df = (
    test_AT_comparison_df.copy()
)

percentage_columns = [
    "accuracy",
    "precision_abnormal",
    "recall_abnormal",
    "f1_abnormal",
    "normal_specificity",
]

for column in percentage_columns:
    display_df[column] = (
        100 * display_df[column]
    ).round(3)

print("\n" + "=" * 85)
print("RÉSULTAT FINAL — VIDEOMAE + AT SUR LE TEST")
print("=" * 85)

print(
    f"Modèle sélectionné : {SELECTED_MODEL_NAME}"
)

print(
    f"AT figé             : {SELECTED_AT:.6f}"
)

print(
    "\nMétriques multiclasse VideoMAE :"
)

print(
    f"Top-1       : "
    f"{multiclass_metrics['top1_accuracy']:.3%}"
)

print(
    f"Top-5       : "
    f"{multiclass_metrics['top5_accuracy']:.3%}"
)

print(
    f"Macro F1    : "
    f"{multiclass_metrics['macro_f1']:.4f}"
)

print(
    f"Weighted F1 : "
    f"{multiclass_metrics['weighted_f1']:.4f}"
)

print(
    "\nComparaison des alertes :"
)

display(
    display_df
)

print(
    "\nDécisions par vidéo :",
    FINAL_DECISIONS_PATH,
)

print(
    "Résumé final        :",
    FINAL_SUMMARY_PATH,
)

print(
    "Comparaison AT      :",
    TEST_COMPARISON_PATH,
)

print(
    "Matrice finale      :",
    FINAL_MATRIX_PNG_PATH,
)

print(
    "\nÉVALUATION FINALE TERMINÉE — "
    "LE TEST N’A PAS SERVI À SÉLECTIONNER "
    "LE MODÈLE OU LE SEUIL"
)